In [1]:
# ============================================================
# WHAT: Load preprocessed dataset files for Random Forest model
#       Loads X_train, X_test, y_train, y_test, label_classes,
#       and feature_names from the processed/ folder
#
# WHY:  Random Forest needs clean, scaled, numeric data to train.
#       Preprocessing is already done — we just load the results.
#       label_classes = converts numbers back to attack names in reports
#       feature_names = needed later for SHAP and LIME explanations
#
# HOW:  Step 1: np.load() → loads .npy arrays (features)
#       Step 2: pd.read_csv() → loads .csv files (labels + names)
#       Step 3: .squeeze() → converts DataFrame to 1D array
#               because model needs flat list, not a table
#               e.g. [[0],[1],[2]] → [0,1,2]
# ============================================================

# rf_ISOT.ipynb
# Random Forest IDS Model — ISOT Drone Dataset
# Attack Types: DoS, Injection, Ip Spoofing, Manipulation, MITM,
#               Password Cracking, Replay, Unauth, Video, Benign

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import time
import json
import os

print("Loading ISOT data...")

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/ISLOT/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test  = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()
label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Classes: {label_classes}")
print(f"Features: {len(feature_names)}")
print("Data loaded!")

Loading ISOT data...
X_train: (205671, 61)
X_test:  (88145, 61)
Classes: ['Benign', 'DoS', 'Injection', 'Ip_Spoofing', 'MITM', 'Manipulation', 'Password_Cracking', 'Replay', 'Unauth', 'Video']
Features: 61
Data loaded!


In [2]:
# ============================================================
# CELL 2: TRAIN RANDOM FOREST MODEL
# ============================================================
# WHAT: Build and train a Random Forest classifier on training data
#
# WHY:  Random Forest is an ensemble model — it builds 100 decision
#       trees and takes a majority vote for each prediction.
#       Like asking 100 security experts — majority vote wins.
#       It is robust, handles class imbalance well, and works
#       well on tabular network traffic data.
#
# HOW:  Step 1: Create RF model with 100 trees
#       Step 2: fit() = train on X_train, y_train
#       Step 3: Record training time for SE metrics
# ============================================================

from sklearn.ensemble import RandomForestClassifier
import time

print("Training Random Forest...")
print("Please wait...")

rf_model = RandomForestClassifier(
    n_estimators=100,  # 100 decision trees in the forest
    random_state=42,   # fixed seed — same result every run (reproducibility)
    n_jobs=-1          # use all CPU cores to speed up training
)

start_time = time.time()
rf_model.fit(X_train, y_train)
end_time = time.time()

training_time = round(end_time - start_time, 2)
print(f"Training complete!")
print(f"Training time: {training_time} seconds")

Training Random Forest...
Please wait...
Training complete!
Training time: 38.74 seconds


In [3]:
# ============================================================
# CELL 3: EVALUATE RANDOM FOREST MODEL
# ============================================================
# WHAT: Measure model performance on test data using
#       Accuracy, Precision, Recall, F1 and classification report
#
# WHY:  We evaluate on TEST data (unseen during training)
#       to measure how well model detects real attacks.
#       weighted average = accounts for class imbalance
#       classification_report = shows F1 per attack type
#       training_time and prediction_time = SE metrics
#
# HOW:  Step 1: predict() → model makes predictions on X_test
#       Step 2: compare predictions vs real labels (y_test)
#       Step 3: calculate Accuracy, Precision, Recall, F1
#       Step 4: print detailed report per attack type
# ============================================================

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report
)

print("Evaluating model on test data...")

start_time = time.time()
y_pred = rf_model.predict(X_test)
end_time = time.time()

prediction_time = round(end_time - start_time, 2)

# calculate overall metrics
acc = accuracy_score(y_test, y_pred)
p   = precision_score(y_test, y_pred, average='weighted')
r   = recall_score(y_test, y_pred, average='weighted')
f1  = f1_score(y_test, y_pred, average='weighted')

print(f"\n=== Random Forest Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Training time:   {training_time}s")
print(f"Prediction time: {prediction_time}s")

print(f"\nDetailed Report:")
# target_names converts numbers back to attack names in report
print(classification_report(y_test, y_pred, target_names=label_classes))

Evaluating model on test data...

=== Random Forest Results ===
Accuracy:  1.0000 (100.00%)
Precision: 1.0000 (100.00%)
Recall:    1.0000 (100.00%)
F1 Score:  1.0000 (100.00%)
Training time:   38.74s
Prediction time: 2.17s

Detailed Report:
                   precision    recall  f1-score   support

           Benign       1.00      1.00      1.00     38421
              DoS       1.00      1.00      1.00     36373
        Injection       0.99      1.00      0.99       142
      Ip_Spoofing       1.00      1.00      1.00       292
             MITM       1.00      1.00      1.00      3154
     Manipulation       1.00      1.00      1.00       183
Password_Cracking       1.00      1.00      1.00      6937
           Replay       1.00      1.00      1.00       176
           Unauth       1.00      1.00      1.00       150
            Video       1.00      1.00      1.00      2317

         accuracy                           1.00     88145
        macro avg       1.00      1.00      1.00 

In [4]:
# ============================================================
# CELL 4: SAVE MODEL AND RESULTS
# ============================================================
# WHAT: Save trained RF model and performance metrics to disk
#
# WHY:  Saving model means we don't need to retrain for SHAP/LIME/PI
#       Saving results as JSON means easy loading for paper tables
#       Same save format across all 5 datasets for consistency
#
# HOW:  Step 1: joblib.dump() → saves trained model
#       Step 2: json.dump() → saves metrics as JSON file
# ============================================================

import joblib
import json
import os

# save trained model — reused by SHAP, LIME, PI notebooks
joblib.dump(rf_model, save_path + "rf_model.joblib")

# save metrics as JSON — used for paper results tables
results = {
    "model": "Random Forest",
    "dataset": save_path.split("/")[-3],  # auto-detect dataset name
    "accuracy": round(acc, 4),
    "precision": round(p, 4),
    "recall": round(r, 4),
    "f1_score": round(f1, 4),
    "training_time_seconds": training_time,
    "prediction_time_seconds": prediction_time
}

with open(save_path + "rf_results.json", "w") as f:
    json.dump(results, f, indent=4)

print(f"Model saved: rf_model.joblib")
print(f"Results saved: rf_results.json")
print(f"\nResults summary:")
for k, v in results.items():
    print(f"  {k}: {v}")

Model saved: rf_model.joblib
Results saved: rf_results.json

Results summary:
  model: Random Forest
  dataset: ISLOT
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1_score: 1.0
  training_time_seconds: 38.74
  prediction_time_seconds: 2.17


## RF Results Summary — ISOT Drone Dataset

| Metric | Value |
|--------|-------|
| Dataset | ISOT Drone Dataset (University of Victoria) |
| Model | Random Forest (100 trees) |
| Train set | 205,671 rows |
| Test set | 88,145 rows |
| Features | 61 network traffic features |
| Accuracy | 100% |
| Precision | 100% |
| Recall | 100% |
| F1 Score | 100% |
| Training time | 38.74s |
| Prediction time | 2.17s |

### Key Finding
RF achieved **near-perfect F1** across all 10 attack types.
Injection F1 = 99% (only 142 test samples) — slight class imbalance effect
but still excellent performance. Network traffic features are highly
discriminative for drone attack detection.